In [ ]:
# %pip uninstall -y numpy pandas scikit-learn tqdm 

In [ ]:
# %pip install -q numpy==1.26.3 pandas==1.5.3 scikit-learn==1.5.2 tqdm dice_ml openml lime shap gower deap --force-reinstall

In [ ]:
from experiments.core.data_fetcher import OpenMLFetcher
from experiments.core.path_utils import path_generator
from experiments.core.data_loader import TabularDataLoader
from experiments.core.data_preprocessing import PreprocessorBuilder
from experiments.models.classification_trainer import SklearnTabularModeler, dynamic_MLP_layers
from experiments.models.autoencoder_trainer import AutoencoderTrainer
from experiments.counterfactuals.cf_generator import CounterfactualGenerator
from experiments.analysis.result_reader import ResultReader

from joblib import Parallel, delayed
import multiprocessing
import pickle
from tqdm import tqdm

n_jobs = multiprocessing.cpu_count() - 2
print(f"Using {n_jobs} cores")

In [ ]:
data_openml = {
    # 'adult': 45068, #48k, 15 features, >50k thu nhập
    # 'GiveMeSomeCredit': 46929, #150k, 11 features, vỡ nợ
    # 'Bank_Customer_Churn': 46911, # 10k, 11features, khách hàng rời bỏ
    'compas-two-years': 42192, # 5k, tái phạm tội
    # 'pima_diabetes': 37, # 9 feats, tiểu đường
    # 'Heart-Disease-Dataset-(Comprehensive)': 43672, #1k1, 12feats, bệnh tim
    # 'mushroom': 24, #23 cat feat, poision or edible, nấm độc
    # 'phoneme': 1489, # 5 numerical feat, phân loại âm vị
    # 'tokyo1': 40705, #45 numerical feat
}


PARAMETERS_GRID ={
    'RF':{'n_estimators': [50,100, 250, 500, 1000],
          'max_depth': [1,2,5,10, 25, None]
          },
    'ANN':{'hidden_layer_sizes':[]}
}

AE_GRID = {'learning_rate_init': [0.05,0.01,0.005,0.001,0.0005,0.0001,0.00005,0.00001]}

CF_WRAPPERS = [
            'monicemultiobjectivegower',
            'monicemultiobjectiveheom',
            'monicesparsprox',
            'moniceproxplaus',
            'monicesparsplaus',
            
            'nicenone',
            'nicespars',
            'niceprox',
            'niceplaus',
            
            'dicerandom',
            'diceextended',
            'moc',
            'geco',
            'lime',
            ]

In [ ]:
def fetch_datasets(dataset_name, dataset_id):
    fetcher = OpenMLFetcher(dataset_name, dataset_id)
    fetcher.save()

In [ ]:
def run_preprocessing(dataset_name):
    preprocessor = PreprocessorBuilder(dataset_name)
    preprocessor.build()

In [ ]:
def run_modeling(dataset_name, model, grid):
    modeler= SklearnTabularModeler(dataset_name, model)
    if model == 'ANN':
        grid['hidden_layer_sizes']= dynamic_MLP_layers(dataset_name,10,1.5)
    modeler.grid_search(grid)
    modeler.save()
    modeler.save_stats()

In [ ]:
def run_ae_modeling(dataset):
    ae_modeler = AutoencoderTrainer(dataset)
    ae_modeler.fit(AE_GRID)

In [ ]:
def run_cf_experiment(dataset_name, model, cf_name):
    experiment = CounterfactualGenerator(dataset_name, model, cf_name)
    return experiment.run_experiment()

In [ ]:
def load_results(dataset_name, model, cf_name):
    paths = path_generator(dataset_name, model, cf_name)
    with open(paths['results'], 'rb') as f:
        results = pickle.load(f)
    
    return results


In [ ]:
# run_cf_experiment('compas-two-years', 'ANN', 'multimonicgower')

In [ ]:
# ret = load_results('compas-two-years', 'ANN', 'monicemultiobjectiveheom')
# len(ret)

In [ ]:
# 1. Fetching, Preprocessing và AE Modeling
# Parallel(n_jobs=n_jobs)(
#     delayed(lambda dataset: (
#         print(f"========== {dataset} =========="),
#         fetch_datasets(dataset, data_openml[dataset]),
#         run_preprocessing(dataset),
#         run_ae_modeling(dataset)
#     ))(dataset)
#     for dataset in data_openml
# )

In [ ]:
# 2. Model Training
# for dataset in data_openml:
#     print(f"========== {dataset} ==========")
#     Parallel(n_jobs=n_jobs)(
#         delayed(lambda model: (
#             print(f"========== {model} =========="),
#             run_modeling(dataset, model, PARAMETERS_GRID[model]),
#             SklearnTabularModeler(dataset, model).display_stats()
#         ))(model)
#         for model in ['RF', 'ANN']
#     )

In [ ]:
# 3. Counterfactual Experiments
# tasks = [
#     (dataset, model, cf_name)
#     for cf_name in CF_WRAPPERS
#     for dataset in data_openml
#     for model in ['RF', 'ANN']
# ]

# Parallel(n_jobs=n_jobs)(
#     delayed(run_cf_experiment)(dataset, model, cf_name)
#     for dataset, model, cf_name in tqdm(tasks, desc="Processing CF_WRAPPERS")
# )

In [ ]:
reader = ResultReader(
    dataset_names=data_openml,
    models=['ANN'],
    cf_names=CF_WRAPPERS
)

In [ ]:
reader.analyze_results()

In [ ]:
# reader.statistical_analysis()

In [ ]:
# ret = load_results('adult', 'ANN', 'monicemultiobjectiveheom')
# len(ret)